# 📖 Lab 2: Fault Tolerance — Pipeline Stages + Retry

**Non-functional requirement:** *Handle failures gracefully, resume without losing progress.*

Lab 1's monolithic crawler loses all progress on failure. The fix: **break into pipelined stages** so each can fail and retry independently.

## 🏗️ Architecture — Before vs After

```
❌ Before (Lab 1):   Queue ──> [fetch + extract + store] ──> single failure = all lost

✅ After (Pipeline):  Frontier ──> URL Fetcher ──> S3 Raw HTML ──> Parser ──> S3 Text
                      Queue         (retries)                      (retries)
                                    ↓ fail?                        ↓ fail?
                                    DLQ (5 retries)                 retry independently
```

## Learning Objectives

- Split the monolithic crawler into fetch + parse stages
- Implement retry with exponential backoff
- Build a dead letter queue for permanent failures
- See how pipeline stages isolate failures

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from collections import deque
from dataclasses import dataclass, field
import time
import random
import hashlib

print("✅ Ready.")

✅ Ready.


## 🔧 Simulating the Pipeline with Queues + Retry

We'll simulate SQS-like queues with Python queues. Each message has a `receive_count` for retry tracking and a `visibility_timeout` for backoff.

In [2]:
@dataclass
class Message:
    url: str
    receive_count: int = 0
    visible_after: float = 0  # timestamp when message becomes visible


class SimulatedQueue:
    """Simulates SQS-like queue with visibility timeout and DLQ."""

    def __init__(self, name: str, max_receives: int = 5):
        self.name = name
        self.messages: list[Message] = []
        self.max_receives = max_receives
        self.dlq: list[Message] = []

    def send(self, url: str):
        self.messages.append(Message(url=url))

    def receive(self) -> Message:
        now = time.time()
        for msg in self.messages:
            if msg.visible_after <= now:
                msg.receive_count += 1
                if msg.receive_count > self.max_receives:
                    self.messages.remove(msg)
                    self.dlq.append(msg)
                    continue
                return msg
        return None

    def delete(self, msg: Message):
        if msg in self.messages:
            self.messages.remove(msg)

    def change_visibility(self, msg: Message, timeout_seconds: float):
        """SQS ChangeMessageVisibility — hides message for N seconds."""
        msg.visible_after = time.time() + timeout_seconds

    def size(self) -> int:
        return len(self.messages)


# Storage simulation (in production: S3 + metadata DB)
storage: dict[str, str] = {}  # url -> HTML or text
metadata: dict[str, dict] = {}  # url -> {status, html_key, text_key}


def fetch_with_retry(url: str) -> str:
    """Stage 1: URL Fetcher — fetch HTML with simulated failures."""
    try:
        resp = requests.get(url, timeout=5, headers={
            "User-Agent": "EducationalCrawlerBot/1.0"
        })
        resp.raise_for_status()
        return resp.text
    except requests.RequestException:
        return None


def parse_html(html: str, url: str) -> dict:
    """Stage 2: Parser — extract text + URLs from stored HTML."""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)

    urls = []
    for link in soup.find_all("a", href=True):
        abs_url = urljoin(url, link["href"])
        parsed = urlparse(abs_url)
        if parsed.scheme in ("http", "https"):
            urls.append(f"{parsed.scheme}://{parsed.netloc}{parsed.path}")

    return {"text": text, "urls": list(set(urls))}


print("✅ Queue, storage, and pipeline stages defined.")

✅ Queue, storage, and pipeline stages defined.


## 🧪 Running the Two-Stage Pipeline

Let's run the full pipeline with a mix of valid URLs and intentionally bad URLs to see retry + DLQ in action.

In [3]:
fetch_queue = SimulatedQueue("fetch_queue", max_receives=3)
parse_queue = SimulatedQueue("parse_queue", max_receives=3)

# Seed URLs: some valid, some will fail (500 error, DNS failure)
seeds = [
    "https://example.com",
    "https://httpbin.org/status/500",              # always returns 500
    "https://httpbin.org/html",
    "https://this-domain-does-not-exist-xyz.com",  # DNS failure
]

for url in seeds:
    fetch_queue.send(url)

print(f"📌 Pipeline started with {fetch_queue.size()} URLs\n")

# ── Stage 1: URL Fetcher with exponential backoff ──
print("═" * 60)
print("  STAGE 1: URL Fetcher")
print("═" * 60)

# We scale real seconds down by 10x so the demo runs fast but backoff
# still grows exponentially (0.2s, 0.4s, 0.8s instead of 2s, 4s, 8s).
BACKOFF_SCALE = 0.1
max_iterations = 30
iteration = 0

while fetch_queue.size() > 0 and iteration < max_iterations:
    msg = fetch_queue.receive()
    if msg is None:
        # Everything is hidden waiting for backoff — sleep a bit and retry.
        time.sleep(0.2)
        continue

    iteration += 1
    print(f"\n  [{iteration}] Fetching: {msg.url} (attempt {msg.receive_count})")

    html = fetch_with_retry(msg.url)

    if html:
        storage[f"html:{msg.url}"] = html
        metadata[msg.url] = {"status": "fetched", "html_key": f"html:{msg.url}"}
        parse_queue.send(msg.url)
        fetch_queue.delete(msg)
        print(f"    ✅ Fetched ({len(html)} bytes) → enqueued for parsing")
    else:
        # Exponential backoff: 2s, 4s, 8s ... (scaled for the demo)
        backoff = 2 ** msg.receive_count
        fetch_queue.change_visibility(msg, backoff * BACKOFF_SCALE)
        print(f"    ❌ Failed → backoff {backoff}s "
              f"(attempt {msg.receive_count}/{fetch_queue.max_receives})")

print(f"\n  📊 Fetch stage complete:")
print(f"    Fetched: {parse_queue.size()}")
print(f"    Dead letter: {len(fetch_queue.dlq)}")
for msg in fetch_queue.dlq:
    print(f"      DLQ: {msg.url} (failed {msg.receive_count} times)")

# ── Stage 2: Parser ──
print(f"\n{'═' * 60}")
print("  STAGE 2: Text & URL Extraction")
print("═" * 60)

discovered_urls = []

while parse_queue.size() > 0:
    msg = parse_queue.receive()
    if msg is None:
        break

    html = storage.get(f"html:{msg.url}", "")
    result = parse_html(html, msg.url)

    storage[f"text:{msg.url}"] = result["text"]
    metadata[msg.url]["status"] = "parsed"
    metadata[msg.url]["text_key"] = f"text:{msg.url}"
    discovered_urls.extend(result["urls"])
    parse_queue.delete(msg)

    print(f"  ✅ Parsed: {msg.url}")
    print(f"     Text: {len(result['text'])} chars | URLs discovered: {len(result['urls'])}")

print(f"\n📊 Pipeline Summary:")
print(f"  Successfully crawled: {len(storage) // 2}")
print(f"  Dead letter (permanent failures): {len(fetch_queue.dlq)}")
print(f"  New URLs discovered: {len(discovered_urls)}")
print(f"\n  💡 Fetch failures didn't affect parsing. Each stage retried independently.")


📌 Pipeline started with 4 URLs

════════════════════════════════════════════════════════════
  STAGE 1: URL Fetcher
════════════════════════════════════════════════════════════

  [1] Fetching: https://example.com (attempt 1)
    ✅ Fetched (528 bytes) → enqueued for parsing

  [2] Fetching: https://httpbin.org/status/500 (attempt 1)


    ❌ Failed → backoff 2s (attempt 1/3)

  [3] Fetching: https://httpbin.org/html (attempt 1)


    ✅ Fetched (3739 bytes) → enqueued for parsing

  [4] Fetching: https://httpbin.org/status/500 (attempt 2)


    ❌ Failed → backoff 4s (attempt 2/3)

  [5] Fetching: https://this-domain-does-not-exist-xyz.com (attempt 1)
    ❌ Failed → backoff 2s (attempt 1/3)



  [6] Fetching: https://this-domain-does-not-exist-xyz.com (attempt 2)
    ❌ Failed → backoff 4s (attempt 2/3)



  [7] Fetching: https://httpbin.org/status/500 (attempt 3)


    ❌ Failed → backoff 8s (attempt 3/3)

  [8] Fetching: https://this-domain-does-not-exist-xyz.com (attempt 3)
    ❌ Failed → backoff 8s (attempt 3/3)



  📊 Fetch stage complete:
    Fetched: 2
    Dead letter: 2
      DLQ: https://httpbin.org/status/500 (failed 4 times)
      DLQ: https://this-domain-does-not-exist-xyz.com (failed 4 times)

════════════════════════════════════════════════════════════
  STAGE 2: Text & URL Extraction
════════════════════════════════════════════════════════════
  ✅ Parsed: https://example.com
     Text: 142 chars | URLs discovered: 1
  ✅ Parsed: https://httpbin.org/html
     Text: 3594 chars | URLs discovered: 0

📊 Pipeline Summary:
  Successfully crawled: 2
  Dead letter (permanent failures): 2
  New URLs discovered: 1

  💡 Fetch failures didn't affect parsing. Each stage retried independently.


## ✅ Summary

| Concept | How |
|---------|-----|
| **Pipeline stages** | Fetch → S3 Raw HTML → Parse → S3 Text. Each stage retries independently. |
| **Exponential backoff** | `2^attempt` seconds between retries. SQS `ChangeMessageVisibility`. |
| **Dead letter queue** | After N failures, message moves to DLQ. Site considered offline. |
| **Metadata DB** | Tracks `{url, status, html_s3_key, text_s3_key}` — resume-safe. |
| **Re-processability** | Change extraction logic → re-parse stored HTML without re-fetching. |

**Next:** Lab 3 — Politeness (robots.txt + domain rate limiting)